# 3.3 — Window Functions

**Chapter 3, section 3.4** (*Window Functions*), and Exercise 7.

**The question this notebook answers:** an ordinary `groupBy` dissolves the rows it summarises.
How do you get the summary *and* keep the row — each rating's rank within its coffee, each
day's change from the day before, a running total, a seven-day moving average — and what does
each of those cost in data movement?

The last part is the one that matters at scale. A window with a `partitionBy` clause shuffles
by that key, exactly as a `groupBy` does. A window *without* one has a single partition by
definition, so Spark must gather every row onto one executor. This notebook shows that
difference in the physical plan, which is where a student can check it for themselves on their
own data.

All data is generated in the notebook from a fixed seed. Runs on a laptop in well under a
minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, random, tempfile, datetime
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-3.3")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


## The two tables

`df` is the coffee-tasting table the chapter uses throughout, extended so that each taster has
rated several coffees and so that there is a tie to break. `sales` is the daily store table
from the chapter's second and third listings: three stores, thirty consecutive days, one
amount per store per day.

In [2]:
# The coffee ratings.  Note the deliberate tie: Chris scores two coffees at 7.
df = spark.createDataFrame(
    [("Chris", "Espresso",  5), ("Chris", "Latte",     7), ("Chris", "Cold Brew", 7),
     ("Chris", "Flat White", 9),
     ("Peter", "Espresso",  8), ("Peter", "Latte",     9), ("Peter", "Cold Brew", 3),
     ("John",  "Espresso",  6), ("John",  "Latte",     4), ("John",  "Cold Brew", 6)],
    ["taster", "coffee", "score"])
df.orderBy("taster", "coffee").show()

+------+----------+-----+
|taster|    coffee|score|
+------+----------+-----+
| Chris| Cold Brew|    7|
| Chris|  Espresso|    5|
| Chris|Flat White|    9|
| Chris|     Latte|    7|
|  John| Cold Brew|    6|
|  John|  Espresso|    6|
|  John|     Latte|    4|
| Peter| Cold Brew|    3|
| Peter|  Espresso|    8|
| Peter|     Latte|    9|
+------+----------+-----+



In [3]:
# Thirty days of sales for three stores, generated from a fixed seed so that every
# number printed below is reproducible.
rng = random.Random(42)
day0 = datetime.date(2025, 6, 1)
rows = []
for store in ["Back Bay", "Fenway", "Kenmore"]:
    base = {"Back Bay": 900, "Fenway": 1400, "Kenmore": 600}[store]
    for d in range(30):
        rows.append((store,
                     day0 + datetime.timedelta(days=d),
                     d,                                   # day_index: used by rangeBetween
                     float(round(base * rng.uniform(0.7, 1.3), 2))))

sales = spark.createDataFrame(rows, ["store", "date", "day_index", "amount"])
sales.printSchema()
sales.orderBy("store", "date").show(6)
print("rows:", sales.count())

root
 |-- store: string (nullable = true)
 |-- date: date (nullable = true)
 |-- day_index: long (nullable = true)
 |-- amount: double (nullable = true)

+--------+----------+---------+-------+
|   store|      date|day_index| amount|
+--------+----------+---------+-------+
|Back Bay|2025-06-01|        0| 975.29|
|Back Bay|2025-06-02|        1| 643.51|
|Back Bay|2025-06-03|        2| 778.52|
|Back Bay|2025-06-04|        3| 750.53|
|Back Bay|2025-06-05|        4|1027.69|
|Back Bay|2025-06-06|        5| 995.42|
+--------+----------+---------+-------+
only showing top 6 rows


rows: 90


## 1. Ranking functions

A `Window` specification has up to three parts: `partitionBy` divides the rows into independent
groups, `orderBy` orders the rows inside each group, and an optional frame narrows the window
to a band around the current row. The function is then applied `over` the specification.

This is the chapter's first listing verbatim: rank each taster's coffees from highest score to
lowest.

In [4]:
w = Window.partitionBy("taster").orderBy(F.desc("score"))
ranked = df.withColumn("rank", F.row_number().over(w))
ranked.orderBy("taster", "rank").show()

+------+----------+-----+----+
|taster|    coffee|score|rank|
+------+----------+-----+----+
| Chris|Flat White|    9|   1|
| Chris|     Latte|    7|   2|
| Chris| Cold Brew|    7|   3|
| Chris|  Espresso|    5|   4|
|  John|  Espresso|    6|   1|
|  John| Cold Brew|    6|   2|
|  John|     Latte|    4|   3|
| Peter|     Latte|    9|   1|
| Peter|  Espresso|    8|   2|
| Peter| Cold Brew|    3|   3|
+------+----------+-----+----+



`row_number` assigns a strict 1, 2, 3, … and breaks ties arbitrarily. `rank` leaves a gap
after a tie; `dense_rank` does not. Chris scored both Latte and Cold Brew at 7, so the three
functions disagree on his third row — which is exactly why the choice among them matters.

In [5]:
three = (df
         .withColumn("row_number", F.row_number().over(w))
         .withColumn("rank",       F.rank().over(w))
         .withColumn("dense_rank", F.dense_rank().over(w)))

three.where(F.col("taster") == "Chris").orderBy("row_number").show()

+------+----------+-----+----------+----+----------+
|taster|    coffee|score|row_number|rank|dense_rank|
+------+----------+-----+----------+----+----------+
| Chris|Flat White|    9|         1|   1|         1|
| Chris|     Latte|    7|         2|   2|         2|
| Chris| Cold Brew|    7|         3|   2|         2|
| Chris|  Espresso|    5|         4|   4|         3|
+------+----------+-----+----------+----+----------+



## 2. Offset functions

`lag` reaches backward to an earlier row of the same partition and `lead` forward to a later
one. This is how a row is compared with its neighbour — the chapter's second listing, each
day's total against the day before, plus a running total.

In [6]:
daily = Window.partitionBy("store").orderBy("date")

report = (sales
          .withColumn("prev_day", F.lag("amount", 1).over(daily))
          .withColumn("change",   F.round(F.col("amount") - F.col("prev_day"), 2))
          .withColumn("running",  F.round(F.sum("amount").over(daily), 2)))

report.where(F.col("store") == "Fenway").orderBy("date").show(8)

+------+----------+---------+-------+--------+-------+--------+
| store|      date|day_index| amount|prev_day| change| running|
+------+----------+---------+-------+--------+-------+--------+
|Fenway|2025-06-01|        0|1657.99|    NULL|   NULL| 1657.99|
|Fenway|2025-06-02|        1|1592.97| 1657.99| -65.02| 3250.96|
|Fenway|2025-06-03|        2|1430.43| 1592.97|-162.54| 4681.39|
|Fenway|2025-06-04|        3|1797.42| 1430.43| 366.99| 6478.81|
|Fenway|2025-06-05|        4|1297.97| 1797.42|-499.45| 7776.78|
|Fenway|2025-06-06|        5|1443.71| 1297.97| 145.74| 9220.49|
|Fenway|2025-06-07|        6| 1676.7| 1443.71| 232.99|10897.19|
|Fenway|2025-06-08|        7|1499.56|  1676.7|-177.14|12396.75|
+------+----------+---------+-------+--------+-------+--------+
only showing top 8 rows


Two things to read out of that table.

The first row of each store has a null `prev_day`, because there is no earlier row in its
partition — `lag` does not reach across a partition boundary. And `running` is a *cumulative*
total, not the partition's grand total: by default an **ordered** window covers every row from
the start of the partition up to and including the current row. That default is what makes an
ordinary aggregate function into a running aggregate.

## 3. The frame clause: running versus rolling

Naming the frame explicitly turns a running aggregate into a rolling one. `rowsBetween` counts
a fixed number of rows on each side; `rangeBetween` spans a range of the ordering *values*
rather than a count of rows. The chapter's third listing is a seven-row moving average: the
current row and the six before it.

In [7]:
week = daily.rowsBetween(-6, Window.currentRow)
report = report.withColumn("avg_7day", F.round(F.avg("amount").over(week), 2))

report.where(F.col("store") == "Fenway").orderBy("date").select(
    "date", "amount", "running", "avg_7day").show(10)

+----------+-------+--------+--------+
|      date| amount| running|avg_7day|
+----------+-------+--------+--------+
|2025-06-01|1657.99| 1657.99| 1657.99|
|2025-06-02|1592.97| 3250.96| 1625.48|
|2025-06-03|1430.43| 4681.39| 1560.46|
|2025-06-04|1797.42| 6478.81|  1619.7|
|2025-06-05|1297.97| 7776.78| 1555.36|
|2025-06-06|1443.71| 9220.49| 1536.75|
|2025-06-07| 1676.7|10897.19| 1556.74|
|2025-06-08|1499.56|12396.75| 1534.11|
|2025-06-09|1703.83|14100.58| 1549.95|
|2025-06-10|1464.98|15565.56| 1554.88|
+----------+-------+--------+--------+
only showing top 10 rows


The first six rows of each store average fewer than seven values, because the frame is
clipped at the start of the partition. That is usually what is wanted; when it is not, the fix
is to filter those rows out afterwards rather than to change the frame.

`rangeBetween` differs when the ordering column has gaps. Ordering by an integer day index,
`rangeBetween(-6, 0)` means "every row whose day index is within 6 of mine", so a missing day
shortens the window rather than pulling in an older row. `rowsBetween(-6, 0)` means "the six
rows before mine" whatever their dates. On the dense series above the two agree exactly; punch
a hole in the series and they part company.

In [8]:
by_index = Window.partitionBy("store").orderBy("day_index")
compare = (sales
           .withColumn("rows_7",  F.round(F.avg("amount").over(by_index.rowsBetween(-6, 0)), 2))
           .withColumn("range_7", F.round(F.avg("amount").over(by_index.rangeBetween(-6, 0)), 2)))

compare.where(F.col("store") == "Fenway").orderBy("day_index").select(
    "day_index", "amount", "rows_7", "range_7").show(10)

+---------+-------+-------+-------+
|day_index| amount| rows_7|range_7|
+---------+-------+-------+-------+
|        0|1657.99|1657.99|1657.99|
|        1|1592.97|1625.48|1625.48|
|        2|1430.43|1560.46|1560.46|
|        3|1797.42| 1619.7| 1619.7|
|        4|1297.97|1555.36|1555.36|
|        5|1443.71|1536.75|1536.75|
|        6| 1676.7|1556.74|1556.74|
|        7|1499.56|1534.11|1534.11|
|        8|1703.83|1549.95|1549.95|
|        9|1464.98|1554.88|1554.88|
+---------+-------+-------+-------+
only showing top 10 rows


In [9]:
# Drop days 3, 4 and 5 from Fenway, leaving a gap in the series.
gapped = sales.where(~((F.col("store") == "Fenway") & F.col("day_index").isin(3, 4, 5)))

gap_cmp = (gapped
           .withColumn("rows_7",  F.round(F.avg("amount").over(by_index.rowsBetween(-6, 0)), 2))
           .withColumn("range_7", F.round(F.avg("amount").over(by_index.rangeBetween(-6, 0)), 2)))

# The two agree up to day 6 and part company from day 7 on.  At day 8, rowsBetween
# counts six rows back regardless of date and so averages days 0, 1, 2, 6, 7, 8;
# rangeBetween honours the gap -- "day_index within 6 of mine" -- and averages only
# days 2, 6, 7 and 8.
gap_cmp.where(F.col("store") == "Fenway").orderBy("day_index").select(
    "day_index", "amount", "rows_7", "range_7").show(10)

+---------+-------+-------+-------+
|day_index| amount| rows_7|range_7|
+---------+-------+-------+-------+
|        0|1657.99|1657.99|1657.99|
|        1|1592.97|1625.48|1625.48|
|        2|1430.43|1560.46|1560.46|
|        6| 1676.7|1589.52|1589.52|
|        7|1499.56|1571.53|1549.92|
|        8|1703.83|1593.58|1577.63|
|        9|1464.98|1575.21|1586.27|
|       10|1571.84| 1562.9|1583.38|
|       11|1018.49|1480.83|1489.23|
|       12|1171.43|1443.83|1443.83|
+---------+-------+-------+-------+
only showing top 10 rows


## 4. What a window costs

Here is the part of section 3.4 that decides whether a job survives contact with real data.

A window **with** a `partitionBy` clause shuffles by that key, in exactly the manner of a
`groupBy`, so that all rows of a partition come to rest on one executor. A window **without**
one has a single partition by definition, so Spark must gather *every row of the dataset* onto
a single executor to compute it.

That is not an assertion to take on trust. It is visible in the physical plan.

In [10]:
partitioned = sales.withColumn("r", F.row_number().over(
    Window.partitionBy("store").orderBy(F.desc("amount"))))

print("=== WITH partitionBy ===")
partitioned.explain()

=== WITH partitionBy ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Window [row_number() windowspecdefinition(store#13, amount#16 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS r#185], [store#13], [amount#16 DESC NULLS LAST]
   +- Sort [store#13 ASC NULLS FIRST, amount#16 DESC NULLS LAST], false, 0
      +- Exchange hashpartitioning(store#13, 200), ENSURE_REQUIREMENTS, [plan_id=458]
         +- Scan ExistingRDD[store#13,date#14,day_index#15L,amount#16]




In [11]:
glob = sales.withColumn("r", F.row_number().over(
    Window.orderBy(F.desc("amount"))))          # no partitionBy -- one global ranking

print("=== WITHOUT partitionBy ===")
glob.explain()

=== WITHOUT partitionBy ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Window [row_number() windowspecdefinition(amount#16 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS r#187], [amount#16 DESC NULLS LAST]
   +- Sort [amount#16 DESC NULLS LAST], false, 0
      +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=469]
         +- Scan ExistingRDD[store#13,date#14,day_index#15L,amount#16]




Read the two `Exchange` lines against each other.

* With `partitionBy`, the plan reads `Exchange hashpartitioning(store, ...)`: the rows are
  distributed across the shuffle partitions by a hash of `store`, so the work divides across
  executors and each executor holds only the stores hashed to it.
* Without it, the plan reads `Exchange SinglePartition`: **every row in the dataset** is routed
  to one partition, and therefore to one task on one executor. Spark warns about this in the
  driver log too — `WARN WindowExec: No Partition Defined for Window operation! Moving all data
  to a single partition, this can cause serious performance degradation.` — but only at log
  level `WARN`; this notebook runs at `ERROR`, so read it out of the plan instead, which is
  where you can check it without re-running the job.

On thirty days of three stores this is invisible. On a table of consequential size that one
executor exhausts its memory or spills heavily to disk, which is why such a job succeeds on a
sample and fails at scale. The remedy, where the problem admits one, is to introduce a
partitioning key that divides the work without changing the result. Where the ranking is
genuinely global the cost is intrinsic, and the analysis is better expressed another way — a
`limit` after an `orderBy` for a top-*k*, for instance, which Spark can answer with a partial
sort on each partition instead of a full global one.

In [12]:
# A global top-3 does not need a global window: orderBy + limit is a different plan.
sales.orderBy(F.desc("amount")).limit(3).explain()
sales.orderBy(F.desc("amount")).limit(3).show()

== Physical Plan ==
TakeOrderedAndProject(limit=3, orderBy=[amount#16 DESC NULLS LAST], output=[store#13,date#14,day_index#15L,amount#16])
+- *(1) Scan ExistingRDD[store#13,date#14,day_index#15L,amount#16]


+------+----------+---------+-------+
| store|      date|day_index| amount|
+------+----------+---------+-------+
|Fenway|2025-06-04|        3|1797.42|
|Fenway|2025-06-24|       23|1766.79|
|Fenway|2025-06-09|        8|1703.83|
+------+----------+---------+-------+



## 5. Window versus `groupBy`

The two are often confused because both shuffle by a key. What separates them is what comes
back: a `groupBy` returns **one row per group**, a window returns **every input row** with an
extra column attached.

In [13]:
print("groupBy: one row per store")
sales.groupBy("store").agg(F.round(F.avg("amount"), 2).alias("avg_amount")).orderBy("store").show()

print("window: all 90 rows, each carrying its store's average")
with_avg = sales.withColumn(
    "store_avg", F.round(F.avg("amount").over(Window.partitionBy("store")), 2))
print("row count:", with_avg.count())
with_avg.orderBy("store", "date").select("store", "date", "amount", "store_avg").show(5)

groupBy: one row per store


+--------+----------+
|   store|avg_amount|
+--------+----------+
|Back Bay|    858.79|
|  Fenway|    1380.6|
| Kenmore|    602.23|
+--------+----------+

window: all 90 rows, each carrying its store's average


row count: 90


+--------+----------+-------+---------+
|   store|      date| amount|store_avg|
+--------+----------+-------+---------+
|Back Bay|2025-06-01| 975.29|   858.79|
|Back Bay|2025-06-02| 643.51|   858.79|
|Back Bay|2025-06-03| 778.52|   858.79|
|Back Bay|2025-06-04| 750.53|   858.79|
|Back Bay|2025-06-05|1027.69|   858.79|
+--------+----------+-------+---------+
only showing top 5 rows


## Conclusion

A window function computes an aggregate over a set of rows related to the current row and
attaches the result to that row instead of replacing it. Its specification has three parts, and
each one has a cost:

* **`partitionBy`** decides the shuffle. With it, the plan shows `hashpartitioning` and the work
  divides. Without it, the plan shows `SinglePartition` and every row of the dataset lands on
  one executor — the single most common way a windowed job succeeds on a sample and dies at
  scale.
* **`orderBy`** is what gives "the previous row" and "the running total so far" a meaning, and
  it is also what turns an ordinary aggregate into a running one.
* **The frame** separates running from rolling. The default for an ordered window is
  cumulative; `rowsBetween` counts rows, `rangeBetween` spans ordering values, and the two
  differ exactly where the series has gaps.

And the distinction to keep: `groupBy` returns one row per group, a window returns every row.
When a question needs both the summary and the detail, that is the whole reason the window
exists.